# 🧠 MeetingMind AI — Kaggle Notebook
### GPU: NVIDIA Tesla T4 x2 | Whisper: `large-v3` (GPU 0) | NLP: `Qwen2.5-7B-Instruct` (GPU 1)

---
**Full Pipeline:**
1. 🎤 **Speech-to-Text** — Faster-Whisper `large-v3` on T4 GPU 0 (float16, beam_size=5)
2. 🧠 **NLP Analysis** — `Qwen/Qwen2.5-7B-Instruct` on T4 GPU 1 (float16, dedicated)
3. 📄 **Report Generation** — JSON + PDF meeting report

> ⚙️ **Before running:** Kaggle → Settings → Accelerator → **GPU T4 x2**


## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install faster-whisper transformers torch huggingface_hub av reportlab colorlog -q
!apt-get install -y ffmpeg -q
print("✅ All dependencies installed!")

## 🔍 Step 2 — Verify GPU Environment

In [ ]:
import torch
print("=" * 55)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name  = torch.cuda.get_device_name(i)
        vram  = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"  GPU {i}: {name} | VRAM: {vram:.1f} GB")
    if torch.cuda.device_count() >= 2:
        print("🎯 T4 x2 detected — GPU 0 → Whisper large-v3 | GPU 1 → Qwen2.5-7B")
else:
    print("❌ No GPU! Enable GPU T4 x2 in Kaggle Settings.")
print("=" * 55)

## 📁 Step 3 — Clone / Upload MeetingMind AI Project

In [ ]:
import os, sys

# ── Option A: Clone from GitHub (if repo is public) ──────────────
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/MeetingMind-AI.git"  # ← Update
PROJECT_DIR     = "/kaggle/working/MeetingMind-AI"

if not os.path.exists(PROJECT_DIR):
    os.system(f"git clone {GITHUB_REPO_URL} {PROJECT_DIR}")
    print(f"✅ Cloned to {PROJECT_DIR}")
else:
    print(f"✅ Already exists: {PROJECT_DIR}")

# ── Option B: Kaggle Dataset ──────────────────────────────────────
# PROJECT_DIR = "/kaggle/input/meetingmind-ai"  # mounted read-only

sys.path.insert(0, PROJECT_DIR)
print("✅ Project added to sys.path")

## ⚙️ Step 4 — Kaggle T4 x2 Optimized Configuration

In [ ]:
import os, sys, torch, multiprocessing

# Expose both T4 GPUs to PyTorch / CUDA
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

NUM_CORES = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"] = str(NUM_CORES)
os.environ["MKL_NUM_THREADS"] = str(NUM_CORES)
torch.set_num_threads(NUM_CORES)

# ─────────────────────────────────────────────────────────────────
# Kaggle T4 x2 — Optimized Model Selection
#
#   GPU 0 (T4, 15 GB) → Faster-Whisper  large-v3         (~10 GB VRAM)
#   GPU 1 (T4, 15 GB) → Qwen2.5-7B-Instruct               (~14 GB VRAM)
#
# ─────────────────────────────────────────────────────────────────
KAGGLE_CONFIG = {
    "WHISPER_MODEL"        : "large-v3",
    "WHISPER_DEVICE"       : "cuda",
    "WHISPER_DEVICE_INDEX" : 0,
    "WHISPER_COMPUTE_TYPE" : "float16",
    "WHISPER_BEAM_SIZE"    : 5,
    "WHISPER_LANGUAGE"     : "en",
    "WHISPER_VAD_FILTER"   : True,
    "WHISPER_MIN_SILENCE_MS": 500,
    "ACTION_MODEL"         : "Qwen/Qwen2.5-7B-Instruct",
    "DEVICE"               : "cuda",
    "HF_DEVICE"            : 1,
    "TORCH_DTYPE"          : torch.float16,
}

print("=" * 60)
print("  MeetingMind AI — Kaggle T4 x2 Configuration")
print("=" * 60)
print(f'  Whisper Model  : {KAGGLE_CONFIG["WHISPER_MODEL"]}')
  print(f'  Whisper Device : GPU {KAGGLE_CONFIG["WHISPER_DEVICE_INDEX"]} (T4 #0) | {KAGGLE_CONFIG["WHISPER_COMPUTE_TYPE"]} | beam_size={KAGGLE_CONFIG["WHISPER_BEAM_SIZE"]}')
print(f'  NLP Model      : {KAGGLE_CONFIG["ACTION_MODEL"]}')
print(f'  NLP Device     : GPU {KAGGLE_CONFIG["HF_DEVICE"]} (T4 #1) | float16')
print("=" * 60)

## 🔧 Step 5 — Patch config.py at Runtime (No Source Edit)

In [ ]:
import ai_engine.config as cfg

# ── Whisper overrides ─────────────────────────────────────────────
cfg.WHISPER_MODEL                 = KAGGLE_CONFIG["WHISPER_MODEL"]
cfg.WHISPER_DEVICE                = KAGGLE_CONFIG["WHISPER_DEVICE"]
cfg.WHISPER_DEVICE_INDEX          = KAGGLE_CONFIG["WHISPER_DEVICE_INDEX"]
cfg.WHISPER_COMPUTE_TYPE          = KAGGLE_CONFIG["WHISPER_COMPUTE_TYPE"]
cfg.WHISPER_BEAM_SIZE             = KAGGLE_CONFIG["WHISPER_BEAM_SIZE"]
cfg.WHISPER_LANGUAGE              = KAGGLE_CONFIG["WHISPER_LANGUAGE"]
cfg.WHISPER_VAD_FILTER            = KAGGLE_CONFIG["WHISPER_VAD_FILTER"]
cfg.WHISPER_MIN_SILENCE_DURATION_MS = KAGGLE_CONFIG["WHISPER_MIN_SILENCE_MS"]

# ── NLP overrides ────────────────────────────────────────────────
cfg.ACTION_MODEL   = KAGGLE_CONFIG["ACTION_MODEL"]
cfg.DEVICE         = KAGGLE_CONFIG["DEVICE"]
cfg.HF_DEVICE      = KAGGLE_CONFIG["HF_DEVICE"]
cfg.TORCH_DTYPE    = KAGGLE_CONFIG["TORCH_DTYPE"]

g0 = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'
g1 = torch.cuda.get_device_name(1) if torch.cuda.device_count() > 1 else 'N/A'
cfg.EXEC_MODE_STR = (
    f'Kaggle T4x2 [Whisper={cfg.WHISPER_MODEL} GPU:0 ({g0})]'
    f' [NLP={cfg.ACTION_MODEL} GPU:1 ({g1})]'
)
print(f"✅ Config patched successfully")
print(f"   Mode: {cfg.EXEC_MODE_STR}")

## 🎤 Step 6 — Load Faster-Whisper large-v3 on GPU 0

In [ ]:
import gc
from ai_engine.speech.whisper_model import WhisperLoader

gc.collect()
torch.cuda.empty_cache()

print("Loading Faster-Whisper large-v3 on GPU 0...")
print("(First run downloads ~3 GB — cached on subsequent runs)\n")

whisper_model = WhisperLoader.get_model()

used  = torch.cuda.memory_allocated(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nGPU 0 — Used: {used:.2f} GB / {total:.1f} GB | Free: {total-used:.2f} GB")
print("✅ Whisper large-v3 ready on GPU 0")

## 🧠 Step 7 — Load Qwen2.5-7B-Instruct on GPU 1

In [ ]:
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

NLP_MODEL_NAME = KAGGLE_CONFIG["ACTION_MODEL"]   # Qwen/Qwen2.5-7B-Instruct
NLP_GPU        = KAGGLE_CONFIG["HF_DEVICE"]      # GPU 1

gc.collect()
torch.cuda.set_device(NLP_GPU)
torch.cuda.empty_cache()

print(f"Loading {NLP_MODEL_NAME} on GPU {NLP_GPU}...")
print("(First run downloads ~14 GB — cached on subsequent runs)\n")

_qwen_tokenizer = AutoTokenizer.from_pretrained(NLP_MODEL_NAME, trust_remote_code=True)

_qwen_model = AutoModelForCausalLM.from_pretrained(
    NLP_MODEL_NAME,
    trust_remote_code=True,
    device_map={"": NLP_GPU},   # Pin entire model to GPU 1
    torch_dtype=torch.float16,
)
_qwen_model.eval()

used  = torch.cuda.memory_allocated(NLP_GPU) / 1024**3
total = torch.cuda.get_device_properties(NLP_GPU).total_memory / 1024**3
print(f"\nGPU 1 — Used: {used:.2f} GB / {total:.1f} GB | Free: {total-used:.2f} GB")
print(f"✅ {NLP_MODEL_NAME} ready on GPU {NLP_GPU}")

## 🔗 Step 8 — Inject Models into MeetingMind Pipeline

In [ ]:
from ai_engine.nlp.action_model import ActionModelLoader

# Inject pre-loaded models into singletons to avoid double-loading
ActionModelLoader._tokenizer  = _qwen_tokenizer
ActionModelLoader._model      = _qwen_model
ActionModelLoader._model_name = KAGGLE_CONFIG["ACTION_MODEL"]

print("✅ Qwen2.5-7B-Instruct injected into ActionModelLoader")
print("✅ Whisper large-v3 cached in WhisperLoader")
print("\n🚀 MeetingMind AI is armed — both models loaded!")

## 📂 Step 9 — Set Your Audio File

In [ ]:
import os

# ── Option A: From a Kaggle Dataset (recommended for large files) ─
AUDIO_FILE  = "/kaggle/input/your-dataset-name/meeting.mp3"  # ← Update path

# ── Option B: Download from a public URL ─────────────────────────
# AUDIO_FILE = "/kaggle/working/meeting.mp3"
# os.system("wget -q -O " + AUDIO_FILE + " https://your-url/meeting.mp3")

OUTPUT_NAME = "meeting_kaggle_001"   # Prefix for all output files

if os.path.exists(AUDIO_FILE):
    size_mb = os.path.getsize(AUDIO_FILE) / (1024 * 1024)
    print(f"✅ Audio found: {AUDIO_FILE}  ({size_mb:.1f} MB)")
else:
    print(f"❌ Not found: {AUDIO_FILE}")
    print("   Update AUDIO_FILE with the correct path above.")

## 🔄 Step 10 — Convert Audio to 16 kHz Mono WAV

In [ ]:
import subprocess

WAV_OUTPUT = f"/kaggle/working/{OUTPUT_NAME}.wav"

cmd = [
    "ffmpeg", "-y", "-i", AUDIO_FILE,
    "-ar", "16000",       # 16 kHz sample rate required by Whisper
    "-ac", "1",           # mono audio
    "-c:a", "pcm_s16le",  # 16-bit PCM WAV
    WAV_OUTPUT,
]
print("Converting to 16 kHz mono WAV via ffmpeg...")
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0 and os.path.exists(WAV_OUTPUT):
    size_mb = os.path.getsize(WAV_OUTPUT) / (1024 * 1024)
    print(f"✅ WAV created: {WAV_OUTPUT}  ({size_mb:.1f} MB)")
else:
    print("❌ Conversion failed!")
    print(result.stderr[-600:] if result.stderr else "No error output")

## 🎙️ Step 11 — Transcribe with Whisper large-v3 (GPU 0, beam_size=5)

In [ ]:
import time, json

print("=" * 60)
print("  Transcribing — Whisper large-v3 | GPU 0 | beam_size=5")
print("=" * 60)

start_t = time.time()
transcript_lines, transcript_json = [], []

segments, info = whisper_model.transcribe(
    WAV_OUTPUT,
    language=KAGGLE_CONFIG["WHISPER_LANGUAGE"],
    beam_size=KAGGLE_CONFIG["WHISPER_BEAM_SIZE"],
    vad_filter=KAGGLE_CONFIG["WHISPER_VAD_FILTER"],
    vad_parameters={"min_silence_duration_ms": KAGGLE_CONFIG["WHISPER_MIN_SILENCE_MS"]},
)

print(f"Language : {info.language}  | Duration : {info.duration:.1f}s  ({info.duration/60:.1f} min)")
print("-" * 60)

for seg in segments:
    text = str(seg.text).strip()
    if text:
        print(f"[{seg.start:.2f} - {seg.end:.2f}]  {text}")
        transcript_lines.append(text)
        transcript_json.append({"start": round(seg.start, 2), "end": round(seg.end, 2), "text": text})

elapsed        = time.time() - start_t
transcript_str = "\n".join(transcript_lines)

os.makedirs("/kaggle/working/output", exist_ok=True)
with open(f"/kaggle/working/output/{OUTPUT_NAME}_transcript.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(transcript_lines))
with open(f"/kaggle/working/output/{OUTPUT_NAME}_transcript.json", "w", encoding="utf-8") as f:
    json.dump(transcript_json, f, indent=4, ensure_ascii=False)

print(f"\n✅ Transcription complete!")
print(f"   Segments        : {len(transcript_lines)}")
print(f"   Total words     : {len(transcript_str.split())}")
print(f"   Processing time : {elapsed:.1f}s  ({elapsed/60:.1f} min)")

## 🧠 Step 12 — Full NLP Analysis with Qwen2.5-7B (GPU 1)

In [ ]:
import gc, json

def safe_empty_cache():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def section(title):
    print(f"\n{"=" * 60}\n  {title}\n{"=" * 60}")

results = {}

# 1. Executive Summary
section("1/7 — Executive Summary")
from ai_engine.nlp.summary_generator import SummaryGenerator
results["summary"] = SummaryGenerator().summarize(transcript_str)
print(results["summary"])
safe_empty_cache()

# 2. Discussion Topics
section("2/7 — Discussion Topics")
from ai_engine.nlp.topic_detector import TopicDetector
results["topics"] = TopicDetector().detect_topics(transcript_str)
print(results["topics"])
safe_empty_cache()

# 3. Key Decisions
section("3/7 — Key Decisions")
from ai_engine.nlp.decision_detector import DecisionDetector
results["decisions"] = DecisionDetector().detect(transcript_str)
print(results["decisions"])
safe_empty_cache()

# 4. Action Items
section("4/7 — Action Items")
from ai_engine.nlp.action_items import ActionItemExtractor
results["actions"] = ActionItemExtractor().extract(transcript_str)
print(results["actions"])
safe_empty_cache()

# 5. Open Questions
section("5/7 — Open Questions")
from ai_engine.nlp.open_questions_detector import OpenQuestionsDetector
results["open_questions"] = OpenQuestionsDetector().detect(transcript_str)
print(results["open_questions"])
safe_empty_cache()

# 6. Key Insights
section("6/7 — Key Insights")
from ai_engine.nlp.key_insights_detector import KeyInsightsDetector
results["key_insights"] = KeyInsightsDetector().detect(transcript_str)
print(results["key_insights"])
safe_empty_cache()

# 7. Key Discussion Points (Timestamped)
section("7/7 — Key Discussion Points (Timestamped)")
from ai_engine.nlp.key_discussion_detector import KeyDiscussionDetector
results["key_discussion"] = KeyDiscussionDetector().detect(transcript_json)
print(json.dumps(results["key_discussion"], indent=2, ensure_ascii=False))
safe_empty_cache()

print("\n✅ All 7 NLP tasks completed successfully!")

## 💾 Step 13 — Save Results & Generate Reports (JSON + PDF)

In [ ]:
from ai_engine.config import (
    SUMMARY_FOLDER, TOPICS_FOLDER, DECISIONS_FOLDER,
    ACTION_ITEMS_FOLDER, OPEN_QUESTIONS_FOLDER,
    KEY_INSIGHTS_FOLDER, KEY_DISCUSSION_FOLDER,
    TRANSCRIPT_FOLDER,
)

def save_to_folder(folder, filename, content):
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / filename
    with open(path, 'w', encoding='utf-8') as f:
        if isinstance(content, str):
            f.write(content)
        else:
            json.dump(content, f, indent=4, ensure_ascii=False)

# Populate pipeline folders for ReportGenerator
save_to_folder(TRANSCRIPT_FOLDER,     f"{OUTPUT_NAME}.txt",                 "\n".join(transcript_lines))
save_to_folder(TRANSCRIPT_FOLDER,     f"{OUTPUT_NAME}.json",                transcript_json)
save_to_folder(SUMMARY_FOLDER,        f"{OUTPUT_NAME}_summary.txt",         results.get("summary",""))
save_to_folder(TOPICS_FOLDER,         f"{OUTPUT_NAME}_topics.txt",          results.get("topics",""))
save_to_folder(DECISIONS_FOLDER,      f"{OUTPUT_NAME}_decisions.txt",       results.get("decisions",""))
save_to_folder(ACTION_ITEMS_FOLDER,   f"{OUTPUT_NAME}_action_items.txt",    results.get("actions",""))
save_to_folder(OPEN_QUESTIONS_FOLDER, f"{OUTPUT_NAME}_open_questions.txt",  results.get("open_questions",""))
save_to_folder(KEY_INSIGHTS_FOLDER,   f"{OUTPUT_NAME}_key_insights.txt",    results.get("key_insights",""))
save_to_folder(KEY_DISCUSSION_FOLDER, f"{OUTPUT_NAME}_key_discussion.json", results.get("key_discussion",[]))

# JSON Report
print("Generating JSON report...")
from ai_engine.report.report_generator import ReportGenerator
ReportGenerator().generate(OUTPUT_NAME)
print("✅ JSON report created")

# PDF Report
print("\nGenerating PDF report...")
try:
    from ai_engine.report.pdf_report_generator import PDFReportGenerator
    PDFReportGenerator().generate(OUTPUT_NAME)
    print("✅ PDF report created")
except Exception as e:
    print(f"⚠️  PDF generation skipped: {e}")

print("\n🎉 MeetingMind AI — Full pipeline complete!")

## 📊 Step 14 — GPU Memory & Output File Summary

In [ ]:
import torch, os

print("=" * 60)
print("  GPU Memory Summary")
print("=" * 60)
roles = ["Whisper large-v3", "Qwen2.5-7B-Instruct"]
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1024**3
    total = torch.cuda.get_device_properties(i).total_memory / 1024**3
    bar   = int((used / total) * 30)
    role  = roles[i] if i < len(roles) else ''
    bar_str  = '█' * bar + '░' * (30 - bar)
    print(f"  GPU {i} [{role}]: {used:.2f}/{total:.1f} GB  [{bar_str}]  {used/total*100:.0f}%")

print("\n" + "=" * 60)
print("  Output Files  →  /kaggle/working/output/")
print("=" * 60)
OUTPUT_DIR = "/kaggle/working/output"
if os.path.isdir(OUTPUT_DIR):
    total_kb = 0
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        fpath = os.path.join(OUTPUT_DIR, fname)
        kb    = os.path.getsize(fpath) / 1024
        total_kb += kb
        icon  = '📄' if fname.endswith('.txt') else '📦' if fname.endswith('.json') else '📋'
        print(f"  {icon}  {fname:<52} {kb:>7.1f} KB")
    print(f"  {"─"*60}")
    print(f"  Total: {len(os.listdir(OUTPUT_DIR))} files  —  {total_kb:.1f} KB")

print("\n💡 Download: Kaggle sidebar → /kaggle/working/output/ → Download")